In [1]:
import mlflow

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()
spark.sql("SELECT 1").show()

+---+
|  1|
+---+
|  1|
+---+



In [3]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("fintrack-fraude")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1792411499623, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792411499623, lifecycle_stage='active', name='fintrack-fraude', tags={}, trace_location=None, workspace='default'>

In [8]:
spark.sql("REFRESH TABLE lake.bronze.transactions")
spark.sql("SELECT count(*) AS nb, max(ingested_at) AS derniere FROM lake.bronze.transactions").show(truncate=False)

+----+-----------------------+
|nb  |derniere               |
+----+-----------------------+
|5253|2026-10-19 15:55:10.022|
+----+-----------------------+



In [12]:
df = spark.table("lake.gold.transactions_ml")
df.groupBy("est_fraude").count().show()

+----------+-----+
|est_fraude|count|
+----------+-----+
|         1|  101|
|         0| 4948|
+----------+-----+



In [13]:
train, test = df.randomSplit([0.8, 0.2], seed=42)
print(train.count(), test.count())


4077 972


In [14]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
 
def entrainer(colonnes, nom):
    # 1. Préparer : regrouper les colonnes dans une colonne « features »
    assembleur = VectorAssembler(inputCols=colonnes, outputCol="features")
    # 2. Choisir l'algorithme : une régression logistique
    algo = LogisticRegression(featuresCol="features", labelCol="est_fraude")
    pipeline = Pipeline(stages=[assembleur, algo])
 
    with mlflow.start_run(run_name=nom):           # début d'un essai dans MLflow
        modele = pipeline.fit(train)                # 3. Entraîner
        predictions = modele.transform(test)        # 4. Prédire sur le test
 
        # 5. Mesurer
        evaluateur = BinaryClassificationEvaluator(labelCol="est_fraude")
        auc = evaluateur.evaluate(predictions)
        fraudes = predictions.filter("est_fraude = 1").count()
        trouvees = predictions.filter("est_fraude = 1 and prediction = 1").count()
        rappel = trouvees / fraudes if fraudes else 0.0
 
        # 6. Noter l'essai dans MLflow
        mlflow.log_param("colonnes", ", ".join(colonnes))
        mlflow.log_metric("auc", auc)
        mlflow.log_metric("rappel", rappel)
 
        print(nom, "→ AUC =", round(auc, 3), "| rappel =", round(rappel, 3))
        predictions.groupBy("est_fraude", "prediction").count() \
                   .orderBy("est_fraude", "prediction").show()
    return modele


In [15]:
entrainer(["heure"], "heure_seule")

heure_seule → AUC = 0.479 | rappel = 0.0
+----------+----------+-----+
|est_fraude|prediction|count|
+----------+----------+-----+
|         0|       0.0|  953|
|         1|       0.0|   19|
+----------+----------+-----+

🏃 View run heure_seule at: http://localhost:5000/#/experiments/1/runs/9d39654cb1b449f3a2f5533fec4d1276
🧪 View experiment at: http://localhost:5000/#/experiments/1


PipelineModel_cbbc284420dc

In [16]:
modele = entrainer(["amount", "heure", "est_etranger", "est_en_ligne"], "quatre_colonnes")

quatre_colonnes → AUC = 1.0 | rappel = 1.0
+----------+----------+-----+
|est_fraude|prediction|count|
+----------+----------+-----+
|         0|       0.0|  953|
|         1|       1.0|   19|
+----------+----------+-----+

🏃 View run quatre_colonnes at: http://localhost:5000/#/experiments/1/runs/4556fc167b7a4ec282481c924cda9305
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [17]:
scores = modele.transform(df).select("transaction_id", "est_fraude", "prediction")
scores.writeTo("lake.gold.scores_fraude").createOrReplace()
 
spark.table("lake.gold.scores_fraude").filter("prediction = 1").show()


+--------------+----------+----------+
|transaction_id|est_fraude|prediction|
+--------------+----------+----------+
|     T00000001|         1|       1.0|
|     T00000064|         1|       1.0|
|     T00000084|         1|       1.0|
|     T00000220|         1|       1.0|
|     T00000292|         1|       1.0|
|     T00000318|         1|       1.0|
|     T00000513|         1|       1.0|
|     T00000524|         1|       1.0|
|     T00000582|         1|       1.0|
|     T00000594|         1|       1.0|
|     T00000664|         1|       1.0|
|     T00000811|         1|       1.0|
|     T00000864|         1|       1.0|
|     T00000887|         1|       1.0|
|     T00000962|         1|       1.0|
|     T00000964|         1|       1.0|
|     T00000991|         1|       1.0|
|     T00001003|         1|       1.0|
|     T00001015|         1|       1.0|
|     T00001041|         1|       1.0|
+--------------+----------+----------+
only showing top 20 rows


In [18]:
spark.sql("""
  SELECT est_fraude, prediction, count(*) AS nb
  FROM lake.gold.scores_fraude
  GROUP BY est_fraude, prediction ORDER BY est_fraude, prediction
""").show()


+----------+----------+----+
|est_fraude|prediction|  nb|
+----------+----------+----+
|         0|       0.0|4948|
|         1|       1.0| 101|
+----------+----------+----+

